# Fine-tune ThaiTrOCR ให้อ่าน "บรรทัดชื่อ" ในสลิปแม่นขึ้น

ข้อมูลเทรน = **ภาพสังเคราะห์** (ชื่อไทย/อังกฤษ วาดด้วยฟอนต์คล้ายแอปธนาคาร บนพื้นคล้ายสลิป) + **บรรทัดชื่อจริง** จากสลิป 260 ใบ (พิกัดฝังในเซลล์ด้านล่าง ตัดจากรูปใน `IMAGE_DIR`)

วัดผลกับ **ชื่อที่ไม่อยู่ในชุดเทรนเลย** (แบ่งตามชื่อ ไม่ใช่ตามรูป) เทียบโมเดลเดิมกับโมเดลที่ fine-tune
- ตั้ง Accelerator = GPU (T4) และเปิด Internet (โหลดโมเดลเดิม ฟอนต์ และรายชื่อจาก pythainlp)
- ผลลัพธ์: โฟลเดอร์ `/kaggle/working/thai-trocr-slip` ใช้แทน `TROCR_MODEL` ใน notebook OCR ได้

In [ ]:
!pip -q install pythainlp

In [ ]:
import base64, gzip, hashlib, io, json, os, random, re, urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFilter, ImageFont, features

IMAGE_DIR = Path("/kaggle/input/datasets/chujikgch/887uu7")      # <- โฟลเดอร์ dataset รูปสลิป (ค้นหาในโฟลเดอร์ย่อยทุกชั้น)
BASE_MODEL = "openthaigpt/thai-trocr"
OUT_DIR = "/kaggle/working/thai-trocr-slip"
N_SYNTH = 8000          # จำนวนภาพสังเคราะห์
REAL_REPEAT = 6         # ใช้บรรทัดจริงซ้ำกี่รอบต่อ epoch (สุ่ม augment ใหม่ทุกครั้ง)
EPOCHS = 4
BATCH = 16
LR = 3e-5
MAX_LEN = 64
SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("raqm (จัดวางสระ/วรรณยุกต์ไทยถูกต้อง):", features.check("raqm"), "| GPU:", torch.cuda.is_available())

In [ ]:
# ฟอนต์ไทยฟรี (OFL) จาก Google Fonts: มีตัวละตินในตัว ใช้วาดชื่ออังกฤษได้ด้วย
FONT_URLS = [
    "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf",
    "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Medium.ttf",
    "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-SemiBold.ttf",
    "https://github.com/google/fonts/raw/main/ofl/prompt/Prompt-Regular.ttf",
    "https://github.com/google/fonts/raw/main/ofl/prompt/Prompt-Medium.ttf",
    "https://github.com/google/fonts/raw/main/ofl/kanit/Kanit-Regular.ttf",
    "https://github.com/google/fonts/raw/main/ofl/kanit/Kanit-Medium.ttf",
    "https://github.com/google/fonts/raw/main/ofl/ibmplexsansthai/IBMPlexSansThai-Regular.ttf",
    "https://github.com/google/fonts/raw/main/ofl/ibmplexsansthai/IBMPlexSansThai-Medium.ttf",
    "https://github.com/google/fonts/raw/main/ofl/notosansthai/NotoSansThai%5Bwdth%2Cwght%5D.ttf",
    "https://github.com/google/fonts/raw/main/ofl/chakrapetch/ChakraPetch-Regular.ttf",
    "https://github.com/google/fonts/raw/main/ofl/mitr/Mitr-Regular.ttf",
]
FONT_DIR = Path("/kaggle/working/fonts")
FONT_DIR.mkdir(parents=True, exist_ok=True)
FONTS = []
for u in FONT_URLS:
    p = FONT_DIR / urllib.request.unquote(u.rsplit("/", 1)[1])
    try:
        if not p.exists():
            urllib.request.urlretrieve(u, p)
        ImageFont.truetype(str(p), 30)
        FONTS.append(str(p))
    except Exception as err:
        print("ข้ามฟอนต์", p.name, repr(err)[:80])
print("ฟอนต์", len(FONTS))

In [ ]:
# บรรทัดชื่อจริง (คีย์ = md5 ของไฟล์รูป, box = [x0, y0, x1, y1] ในพิกัดรูปจริง, text = ข้อความในบรรทัดตามเฉลย)
_REAL_B64 = "H4sIADEI2GoC/819fY+ct5HnV2n4ryQYGySLrCL3v7Gti5XoxZGU+ILFIuDr2sDGDrxaZIPFAZHPgJw33N7FPlv2nRNZZzhnQwcnlrGjb6OPckU+3VL3dI/64czTrYEjqUczURWL9fKrYrH49//23M+Tee7vngNhozIyOnI+QspJYbE+FioZFEj33MFz5Y38T4l/tLz91s9/9qb/eea/C2/963N/9/ca4EALOEBFB1rjPxw8dzP/603+0UdHdx4d/e3R0WezR0cfPTr69tHRvUdHnz46+uujo/9Rv/fw97NHD2+98Nx/OTgFHzffWuVCOndgjGQu3AEKWObii0by/UdH77UPzNSHja977fe7s+88evjrR0e3Hx09WHzzm0dHHz86Onp09IdHR//r0cP32g/eaT/2xaOHt+t3H7776Oh3j44+4FV8d2kFolhZPAEYF1y0QvpUbBRUgkAoaJ4qSSA8ACMOjKYDsPYskuzg47gktcADbSsXLFElOiS5TB8k7wRaZZKhooWJIABIBqOllzZs0Si9pFHmTHIYz8dxOSghljRKPyuNktKbZC0YL1IEcso5w+txCYqIWcp9SbKDj3Nqm1KTiS7FbLyTUnvjlLRYsLBeKGtoX16ug49zqpMBs9LITJAtUMgHsEFap9jVFMCo9qWTHXycqJMgnqVOImuA0VlbGZRXLtjEYgEttKAQkyj7kmQHH+fUujXFlNjZI9kAmrxw7JgoKJVN8MGVDus+kyQ7+Din1q1dcUE4BzklMGAxFuFFiAYYP4gtsXtCDNPBx24wTLGFAspIjlLwiIlNROeUo2HfnWhvGGY8H+fUNr3TmdgAbOCNtFpELQUKJUBo0sHDviJvBx/n1DYtmpSUVdkKlaK1JvAfkaNdVqClHSNJOZcknUWSHXycU51UqTgviw3CAkTOldCaKIXzxfHfSdyXdXfwcU51MrJZJUxe8+9RygREIJyKLnPSSVbuK/J28HFe0aBVMUerdJG25GxiypCsiyYpsi6kLZI0k6HB8XycU530FLWNloT2NhROqyKjW4EaPUMDQPt0SbKPBJIHhhGElupMEWc8H8clCWgODOMYY/mXWdHJ3zaSny1E+pj2nD0WyZeL733ZJP6Av73EFfMQMiCxwxE+R5SoddCS8SmG4DCPRHh4VoTXwcdOEJ7mPbHaKJ+ig+QlFfSYGWBx8ogoU0cMOBMu6eDjnNqbETk7V2RmXKqzVJYBVgbQQSoIFveWx3bwcV6jacxBsr+IESlaz1iAtPNZELhio8r70skOPs4pwuOwr9Ehe3LtIpbISMuy02KwxWhfB7uvrKODj3Oqk6RctMmIGL3xwpQIhjMpVYRxklMAty/r7uDjnOqkDeAzJjTSsINn4E+E2XqVixWBfdcISaq2Bq31mfK38Xys4xI4MMg4mS3EuJMtg7+4NfxeiTckwpL5TZM3/9Tt5XM/bbMLhcF7DpoixKzIWYYCRB4Yz+9Lvzr4OKf6RUnHlKKwWWIGTyVoZyy7bC0xlWBGSFJP4fM6+DinkozKsT3IRNIlZHSKLZUsQubiXTFhX9Gjg49zmtWWkkCZkqGY4j0jW0oZlSyMzjhPj7Av6+7g45zqJCQDktAE55IGUSIjC4EhZeXZvMz+/OR4Ps4poknk0HsdJYOHGDSvgBxjMaOSwwA2bc+Aa97ZzjjOVEft4GNzBiznGbA8XQbsJdtABDSIhqxDzb45Kg8ZvSJFcYscaKqzng4+dnTWo4tXAsgZhdJwrmisFOxhlQoold+Xt+/g47zmr5zacOrtSZWImjdRCXYLVqI1nD4S7i1ujufjvPqoUh0D+VCMoJyKRKMFJ5Oc7SghrR6ZK5z5rKeDj/N6/liiNjpw9CcDJiQix/kOOxvGABLI7utEu4OPdS9HZ/dyID1q4g0BQ6IeJzurcgg+JfAMgmBfJzUdfJxTTBukUaXIzK5FSYc2SVtAMijwlv+ecF9Vug4+zu05LChVgmMcLoWTmnGAFjZS4WQ8kcd9nUV08LETBEJGCopsEUowDnIi2sKsECfSivdU5L3V2MbzcV5rbBKlV8XqUjh5Bgv1jCmqwmtiGGXSvhBIBx/n9RQVk0GF1jvJUMpGdCny7wZ5p3kFZW99O+P5OK9nEaSCNmh8to5SolKSLzkwvg8lZb036+7g45xKMukkBEP6UDx4R1LrohQGhlUxazcqv6hrsGdGxeP5OK/VJM6GYiAbY8yYrUAVvRDBkygh6xRHWrc9862f8Xyc11s/2XlfvPWmBK21SNFTsCIlCxAU+L11io7n45xK0vjEMLZw1m45S3LaOpAhFeU1Ys4kR+a89qznYx18nNd7Q2gFUYm1ZTqVopW1TmnkpN0Yx4va272h8Xyc1+oBm1WRDH+Li8GRV5I8acc5q3cQrd+SoZilWvHZqgfj+dhJrVhhiUVTQSFDoeizgxCc9UkEa018+tmiQs7QwLWeOqAlOdzI+W3/i9f9zdl1/+Y/xtf9L/0LL7xwOrJrpoiOl62ZqDowcskUXzu8dnj9cHb98Mr3X3rl8LXDVYqQKkj3nEYHAFlP3RJng5kcA09bDGxZKC8QmB7qA0DTsdAOsmsLNbxQEvyLDoxYykB/cOH69cOXD2evXL3y/evLHWfoGUy7EF12VgTDv8cUIqe+iZOV8nQUpaouS9Ym6Vi3dccSO8iuLVEbdkXsWiXVZS4Rfftf8uW33sy/ml1//a1fsOWz6j78zWDexzRJ5BhE4LRMl3aHURdWRxWlz5GS2rKvJAYFNpYVGHsUeDzZtUXXPa3KW3supOtRphBCygBJCkuc0SfJDpcxPkcnr5KyO1prB9np1qq9QBYtOwkVc1ZeCoORPGbJZHFL2/Tp19pBdrq1GiOiKyxm1h0TQ0Q0xsukIklSUNQW38SGW22oOWHcDdmTnTD0OGH2ekoKpsaepqjWJiwjsdEwvC3KPT1jVoajjWIPxY4YTM9CO8iuO2GGnYYdcE2Nlo+FL1+7/sLVa7zCK7PXDm+89Mox0SpXUiqS0aUMCpNKhUG7C6JkgeT9lh0F1t5aeuQdRerZ0fFk13eU0SFaJsoaLOTyjvLqDq9dvPLq1WtXeGNXKeZYnDKlkFZQ72ZYU7ws0Rqfi00aOxbas6MdZNcXyjGnqS//cstm+srhlcPvH165+kNe5bJLCMUlcEEFz76gpg8VsiRvgRMJtyV9qdapqqFo9kRK9Xii8WTXl8jhVPF+OkaIK5ZywhJLTYx8EcZSVgSJ8YkwkI3QSkCWYkd22UF2IrtMnvFXRJnZMgAdajCxgE0m69qBqrdEFTl4WgZmnAJ0LLSD7HpU0UNU4T+NXMqXbly4cO3w1VcObyyc7THZlmAJPFqOZVYghOxFZCUyoA3DFtjRSjvITrVS41WA4oQOOWGtk1A0Viu2uhgko+0dIYUOstMhBbYSyJkKe07MkR28ILS6IMSSLHuIXaHd8WSnW2vOGktOjDG95AjO8KsU4QhViSZgcjtaawfZ6daaXPIaooy+eEUOWJkia5fW0TgEtcUv1dgi2GgUO2DA3ZBdT904xtSUWLIjxpU6zq1WaLjbPgxFh1utbPNlK9V8Plv8xNetLHGvlStuH3MjFIITwaIstqTsa3YVTSJDAYMWu8rqOshOt/f8T5N1WheIjlTiXIMRjc0qB8NgXNtxJRhynSWYDrLr+MIO6J+Q17oUHi6//cLsxutv3Hzjlz7Nrvg3b/pfvnHcvEq9g+QFeUcQSLqaLfMf2STr/JZrJ4oFDKJe4rR11U9b7ampri2WUxxd2xFEDU/yhLC0ctGWUSgSsTmiAJFitgmi0UwbbClyKt1dXWEH1TOo7rG8yiuprDQkHOM4iOwPjEmohVSM5bY03Kra3lELTkp0Fpw6yK57LWwFJ11r4LRSx7/fnBV7pq+fZyfVvni/FW8fV56+WOnplFIKL5SWQsbglPdGEkaRMEuUNFXKvrryDqobc4KardeelhWbvf7C7IeHP7l443B2ePny4Y0bhz89FiRyTDIYrLdnNEM5b6NKQLVqzYhW5nFIslZQn44kT011KiApvXIc8k09tGTvz6gu+xw8AIN3X0zajeF2UJ3McDXDVt4VmwKxswg66qQzu31bmBstd7TSDqqTrVSmrDnvCC6GzCG9ZCU5QxEheha0s1uCKyeYIzPbU1PdWPVviW2NsTi2DpM855gqGe0NIzoKYArIwsL1nHYqo3ezox1UJ9vRAnXSXlHsyKSVrE2es82IjNFTVuxmpiqtnZrq5sqaayWglYLT00sVuYCRQljt6njPaLFmYNKzK0ysFyLuCiiNproRKNWzyLFASZmUc41drsSIjuWjSXDyXAroHKls2UnNK5NjaofHctnxVNd3Euc7KVdP5J5um+RIZ+dIyJIZdEhNKUIugX2gl8XncRrL9tmlsR1Uz14iLSk6r0MyxoPnLMJLAZ69AqZ22WdXhzXjyU54CMdU2LcTaMxM11B2UXhtbbGSnLG70doOqiftpuGocsz/cLbWbJNN86VXrt64+OIx+ZrA1uJKTpxLQKj2wR4vk48QrFR6N8ivg+pUyC8EtLpiaIOoOWAH1p4qXOtY5H7LrZyOLohTU93oiaq/tWb1/O3GhXrsdvHG7Cq7pGVvIKVirGVJO+MCKMzCoAqG/R6jMe1GFpj0tgLTMR80nurG+pJSrR9B40q30uetqeXdlqx9y8nZbFFaevL1F/zp+VaDagWoh79vPUG3Wk/QB7PH+d0xG/MM1gqZGvek0SA4o7I25sTRT8m4IwzVQXUyDEVWJDIeJZQ66dkwaW2EMoxvilCYR+q77NP3DqqbK07QGnBW9P364fWLr128fPXS7Ac/vsI+7PLKvAKWLQgHSjB4q3doJCdYnG0xTcNYdSSCcn0IqoPqmUtNwXCCbAwnxtXAIJmaSaJS0YHxZewxc0MWHRrbQXV9H2GIRQJWEdT1i9cuXjtk3zV76ers0o2Xl+OBAk6OBSbOGEO05NB6z1g0yUIUjJjqJOtYFBpPdaoolCTn/kmowKEglTowzZKgIIgBjRNyR+G2g+rmhcJQCz5xoSvxIIkMnDD5EKRKHuqwBmsEU/VZJrWjtogOsmeGwtnEehuZ//EQipdSWpeEdSUUx35gW9p2+lOr8WSng8ICMxVHXqRCWM+LOEukHDOHLRNZCcetFXv7ljrInlljRVC6+Dq6IaesY7GJKih1wRBHiBB2pLEdZE8uN1BPuaEYT3UGN0ZR2OrBYTEqs3gZtvE6YEcL7SB7ZtM0MgTWEptJF+EwZhuMrlM7bbRebXnhoNblWubWWoC7mrLGk91YDMShDXilKesHF6u2/vjy7AcXXvvBsqcrXmiot1tVSkYjp4vO5FRdXj0kESO7nKGzy7mD7PouymF5FdktL5Gz01evXr1ypWUzL/Fajx0CIXAs9j5HrZUWSRcVpZAs4KAYYOapCrv+1GRPruzSasvS06tHLqDi9DdFTdJ6mXNAQZFzqeiE86h3tdDxZNcXygus/VlkVxd6omEqyykiB2RvhEUZZKwvf2ROqEVx6J8eM6WtboC11tQEFTZcxmh3MR7UOyD197Xxut/73vdOx8va1RZ2RMReF4U5sPKEq0pDpvppu5qxuBDSbqhwrvpVY/BeY/APy5eOk+Kd8Mg5M6fRyYpC4DSDMzYahmlPVwFJHO7qcEFZrxKsyKdDLB0srImF1DB12dYJDyu3Cj9pR6v3mzCGnXpn8eFO+9Y3iys/37YvP1s9hGXnyaYojTBRBq1D1jkkw7Ek5yhhy6MWT/TGTKE3Hbys641mvRlGU1vhNuvNJ4uumT8O5D9ZtNB8NNyKWAYPJmTB9moYw7e7wVkKo2Xtaclg1RZzqv645p9UvZU+q1g6eFkTC/tL1OZAORaPXNGboW3oYTOa4X7TnaYi763MPMgx1cdasL4d5DPDJkuczrk6b5S2NNVPrBwdvKwrB7By1Lkojp3KyhyKOy+wO3lh0IU7ixtf/L+/DQWwB+3u2+eNxa8XLC7rSZSKA5mFVDgsIWfuJYgETgjI9Wap26fb7eBlXULIElIH6CxLSG2U0J3Gw/99dPTvi660O8OdwFnzPQ+qWc370R407/zZseHqtUEqFEhaWZ2cDzIX7Yk/UYmxPL3QIustJ6lYXK3F/uwRajQva6JiTrC2vwCwuJaU6UX/dshvz17xb7w9e+lfbi7jAIbX7L8Ke7aitcxEQUQ0pVgLjrHsPp1JBy8TOxPrFCaMDPhszCz9gC5ZMj5adtgG05YIXJuOaqdTzcjorFLo4GU9FLOpiPb0wAGhXj0Kun71x5cu/vRwdv3ipVdfuXDl+5wNX3llmS4rGYJTMjHOL0nJLFUGZx1DfItbDkme2ICZwgY6eFm3AWRNqIkc24Jbclzfv3b44o3D/3xx9h3GqRcvHV55eWXYB6dsLiasxXpRFAtd60IhQnJOC79XO+jgZWI7KB5FZjScDWcFobZO6mhylT7ntEhb7ADwAFxtEqhX+d1ZpdDBy5oUgKVQew8d68ByMn/p8LXrV69IYX/2KpCE5394+eKNS7OLLy8fpYkUZRSKzY1sKCZg8aBCIuVFcIByn8Cig5fNwII5cRVYrHiluw1yvzv0aj9+l2TIXO4sFOXThZZ8VP9+JVqGkCEHDlNJGA/RqACisKMySfO3ttxTfaIlZgot6eBl3VNwlJS6Jc+IS8DipQtXblw7vDT7T1evvjy7duEGe4vZS1cPXrh042V2Hm/94p9nyb/xT79ameHtXS0UGOmiAJGt8Anqo72Mj72nLW98PREJTSGSDl7WRaImEwmrbhFUik9O2sTI2COQsyVZJWjPdtTBy7od2QMysp0wWWE2cnJnaQzDcU6+WHyPkfu3yzAv14sZRdjIqbZmPlKKCkNtIhSWMe8+xdPBy7p4TEtuSYidoXN2ewCcXRIyTyZQipxWeRFJQJ0RK/cZmzt4mTg2o08FZdDJ+hg5h5JRByAdEyMCVmnYp8J08LLzaogqImgoSYKKRMXkxH9IJEO5OL0FsnAI0AwUBth6ZujewctG2ArEsJX/dGYsbAVZomB8xBAhUfCMHOvhbj1PchbllolBkumBM3PAdubVd/CyGbCJ4RctnRJcunjlwuxV/6vZ5fx2fN2/eXP2nR9dm4VfzW7eDCvzIzlVFj64nJRQrfxUGDK6aIxMwoq9wvcOXiZ2ESKLejU112c+Y0nkEzF89IpFarLacmA7NXQdz8u6iyAOuawJbJ1Wyo0uotaNW4m9xo57c0C7CLj8xV9qU1jl9KjNjVr2GIE9da2u1FFh9Xok22ydOk66JChmW6I7rZQ6eNkMTMTkwCRrKhmzEEEkZQg4uKFirCSDU6IItd9MeDQvE5sS5PqwkvFBsyIznlYJvOc8isMX5xukRlSExFQVoQ5eNleEaF4RMn0VIXCFw7nhxDPZnLNSIWbOriwJl4wOuM+KUAcv66FVt6oocphbqYreuPrq9dnLHFR/+jMOuer5H16a/eTipUvLR5cV3LBhqhiM5FgGxL6rvipF9fUyseU2igQ7j684RXzt4GWXqa6QSjP4kzaojN5xRNOOcbDRjIw92L3i0A5eTsKhiHQyDn0qKwvn8UH98WEmYR0J+NnqO2kUI/C2MSpyoriMNShKIQwjIkbPW16kr2erGgS71Nokok55uNvBwubD3aEtBvUKA18t+q8/X0xm/LCyUT/fHuLzV4tG7P9XvW39xm/blx/zl7NWeltx9YwSOPpZpCh1xBQNqdpSrqOt14D3eVrVwcvJ53l04nnevQUquX0s5x3y4W8XMy7fW+jT8qF4ssZmkEk5Ve8gliJ1TnUvTZYhJ7PP8NzBy8ThOVltFQrlONHSQgXnsrQxpcChMaQtjVCStaTOtmihyaqzS2E0L+tSYE2pfZksDVqe8Xn9pRdnTQCMbr+dXX/l6qt1SGIzn4f/tRVPfvPd1euLMggVWfpOQ+CkC5LJxuR6m3rr6cW0PrmDl80+WTFiUWvG89gnf7qE748XqWeL9poP2t+22/3NUS17wyyjkk6SKwVCu8+qrDIx1MvLnKvs09F08LLR0UAd6KY3J0m3BwX6rJX177W2gYfvNJl8NDwuXoU4b0h6sDK7FZITsiiNIVH0po6rjTFq8vXVPdpr1814XjYWJsEcoKq6ZDY64qVi5HzoMDPz70Pweuex822fv1nmqh7Lglds4xA5ryfrfECHinVc+Bz26YI7eJnaBRvORTgrkxhdsom0j4k3SHoiS35UhjTZmXkHLyefmev+M3NWT2+J1auADXWegxL19N5aGzz/LvZakx3Py+aCi2kozwpat5WGRe4P/vfWYs70nXZmeGuY+rSMniSvRnn0sk7VyoI9FYcD1sdgbHEh7BXJjedlY31FV+VY7ztarkJ91b7+jyafJ/IYnO27jc3jQ8KXryFZFNkECKST1Eow6nSc16ZgmdlMekR+ie1OmdZnTrE7eFmv39qz1W+1srzjlGIImVN6C1kJCFbXMYqWIeUWndGDztRkReGZ5TCel83oX7YJ+BuTyttzrL9Ur3yv3WD99VDIXO6GEp5TWxFCCrwrbMRFUOaNyLXnklka2RUMZ0gcO1hYd6umvjReZ5wcIBwPc3ebqSzmL9WDnTvNWFbWz+C5ZIoGQnbFJSIfc5HAUDIoGeI+e1E6eJn8aKNeqbUyah1Z/Fk5dukiOWIrJRey2OYiDMtBTSWHDl4mdxEuCieDI6myQ0aBmNEwJKzlHo5faa/trB28bMbtjEtrWDnmIlYPzD9oXNVu39+0r+82L/Hlol/+vfbt5cyGU/HM6YBO9S4HZW191AAuIxgOd3qvyL2Dl5NaCtCJtcxmIaE/N+ofDinxbCW+zie7fbN8LLTs3TP5oCBUrKxIg3daCi3Zw1kIacurXkPlYH5ebM8M3jt42Vw50O11EFJLnLAl/TyzEd18q72tkPOy2hoVg49CFKYmQVkbpZS6GODkSm0ZpTt1s+94XiZOXGQI1iXjPNQBt9KBjJnDW3vL2hXI+zSTDl42F0tqWbaC9pNfqbn9uMfvmJ3cb59vtc6c5cCbVH2MVLtUYgwc5YyUBQQHPSm8E3GLo0WxpCVnxmIdvEx9ll5ssa52YWtdkEFPqu3oviRbOLE0ep9XAjp4mfLwKziRkCxDPcQc2DfpCBk0Ry+MOaq93r/q4OWkSllFHscN5YwRF0q9vMK5tvDgEa1n4F9qv4+RPpcURh4P0iTtN+N52eXxoFFZ1yJmqrNpOaGs1whZeWtZKkixZcrm1H2N43k5GabR1DDNeuEpSqN4W4IHykZYk+oYN7C47ZGCiSXUwcvm+MOKo3GtaHSm+MPKG3RkVupRkk+MBVTEiBwNExCj7O0npjXxbSemp70O28HCeuILnPgOIygQlhi4/sqPX/zx7NXDazd+ugyDEiptQ4qRnM5OSiokbQxSFUOU7D7bFzt4WfcgNNy6EbV9cfma/8UrV6/NXv7R7NLFyxdvXFge/GOSB1l8ktIhBSWizpnYSIvlTMpT3ufSO3jZeOmuv3OThAvKahTAtItNGZJj6bOy5VCHkY5cvZ5i9R28nHzdCnuuWxmRsnGFyLgcWLec0Lb44rIDU5QuW1bvDlTLxKjOrdvsef7cPg0F2Xu1hfh01NfXa4d3lOpEHRDH5tbdbw9T/r7RrcDy+XbNaBgL8D8XvQ9/XZzvf9iS03fal++0//9n7dDp0+UUUaAIWMe0Gs3bojSRCN4IjujGuLQlWHCgqIOjdK2GbQwWWwTVQX1NUDVI1VoP1nOFJbP452Egxdtv5Td/8fpbteLjV0YO1ylVnAUJUDKleu/JGkpYB4N72tbOzSlqG3cEmtcr+tfbQX3jeueddgZPoH1v0QJzZ96V2axx0InbTSdqw8yyZSqUhYoLOblSi29FuPpMVBAMg4nGiGOx/aZfHB3U18Whmzj0cHVm6Sqyus4Q8oUDhpArI7VN5Bw4MkBTngFIdlbFUCglJuZTDCP33Zxu3zuob973WuRu4wtH0h5mgLTi/+dLMOiPx0+HlKqTmZPJouhCQBEAPGiOVkFoPU4oWq8+TTba+MdTXxfKfMphnYezbAy1k+XChVcPf7pypcY4h0hCilpIJiMEO2ZhyFkBVPI4F6fN6VxcB/X1VSKvsp471flGy1G4WvL9No30bnPt99sUmPvDueCvH/d7tc93H7uC5cNcznbBsa/lyCzRWO+FZPRZ7/TJqLegA8VxUemKDPBAWeqXSQf1teFA7GtMPTWl2gO3YoofD60mTdFr8XbW5uL89cmLzvNm9AdtFsy8h3BzjByOzb5pzNe5GLU5ahiw87fmYe+s3JFRVubCQEHZEkvMrCYSyKf6DJ3X26ahnlWWHdQ3y9LMZSnPgywtO8j6Mn2q7/klyOhlKB59dgW0MFvm5TXcVs8E67xh2S/LDurrQ6vYRttLy/WF9KVq4Is3rs9uXLw8u/GT5dmySosIgQJkCygDqJzqM1EkogjSkN/tOjuob16nq2vkkLRE+0fXZoujAX9zxqte7iyPpeTsFWSZDBioDVZBxFjn2qawtaGJPXAdYlfLWeoUq+2gvjn42vaahsETrPPD5nlvNxf8/tAX99vWkPzr9uE/hiFVK11nxpEoAYWCaKIpRhRX77B5LxkFlq3DtM+2+eOprw/Vdm0qLSGt1h6+M49IR79VQn63NWQ/aAfuDxZVLP58NPSs1HFCi+6VeXz6+vn26Xcc0ZY9GwXMFsFQzsVTCoBOB6cTkOI0Ou3WRjqob5o9Xn2BlbTqCzb61aE/4bO5Eg1N7tXBtrHjR+/M+6CGtney7rg7Xa6vODKQfbbO2GQjRMwxRWlLiVSwlK3PsKoGquEA5Ckwbgf1Ta+xslIegKuHCd1HKRkxsANjyJg44tWHzACUZ9TGiq3LtlmOZ1WUDuqb7YnaRURUm4vA99vS7600Hrexf4sLEPPLEbcWLH67Aj4B64PO3llSGSxEqQ2jTuMZJii5YxPqoH6SZOqwS9zYcjVrXqZdD6n29KehCDJclGHTYcf79crM8er2jZOkKEMRoY4aD8VII7MUW0aKnF0Q46mvC8IOKiLdqstd7hJ8pwWad5cK4XP89VXzLUMT1PIRQRAMAbxIhlI2tfcsqywERsNJeMpbemvPLo7x1DfrBQ7HjXA86by3CDgfLtznvUXy+4ehFPLRnMe5yD5YxKZv24+umI7mNeakQROn7ezGyJGVznGKKh37uR2LaDz1jSKqr/QRbDad28M5yf3hZtmstenfayIYJv0dKxG5nFlfKbnMGkxZO5RIJDWGULQvuFtBdFDfDFVte4JhXVfmtO+3L47WS9j3m2P9pqG694YsZ7nCyxm8CdG4EC0DaEeSHZrIqJO2Wae4W6F0UD9JKJx1nyyUU5aTgmCcHCyaOkidU3zpQ3KB83mfJQkou62tdlBfh/lyqK1asVpMXDaZLxavEc896qw5jttNKH9d9jrL2muNqygbJCZfIMhg6mMlnJb7hAg7Djsd1Df7WVNbv9aQyaLR+MvhVdPBm7SDaf5Qne7jaSp/HvzLZ4s6wdCW+6D9v79alRQhaKuDYFsPxUotyQPqUvuXOPHa9urtWSXVQX3doOQQkWrD3HIj8uO+083tppxUSGuyDdqKJKkOImRi/C85Tk5t2HF86aB+olt1YnK3SjFYKVV0LllWXQ57XhKYpBk+xoI7xiUd1DfbSxWKXYNp8xsuw/y7xZXmVhe7v4ZXhh/682oEjkF4q0DXx6fJJUFagbRQ5zP4YEPZrVQ6qO9VKoFZAVZd5UsKEpEc59uYSWDxQUW5vY1Qmfb4g9kolcXosAFB/rc6XGwYFXWsb2w8F5uKS1TLha0EayYNxbJUMOCF9MnH5BSiVzICMpjC+qjsblWmg/rmfGcO8k/Id+aA9W/Nu7BifPTYrwxu5701MA8oFHH6xQEQo0BOzcEV3iIdif1+2nFZuYP6Zgty7QjoRGfb5pgtNUG93womj+tK/LcfL4NHm7Ko0CgkKC6C4UxUuMjhoEaAbc+anbVK1EF96ioRGZOYTA6+Tq/0gqGyqK+f879Zp8zvOOCOp36SF91ZlUgFy1lWcTKXwOl3LMn7YDg3J6053XK7xe0d1DcfA2Mbb3D8bPzM8UWA1mRjiUV66UxmDq1PFnO9Tw+4Y4DWQf3EqqKptbMTDi3uzueAzJXi3vCOyCeLyb3fzOto83unyxgte2+Ct5k5421jrnQh6Upt6kRNkXZsR+Opn5TToFPHrwzemg9NGSY/zKshdx+Pm1mI67eL90Y++U4T16127PNu06TPVuZlWCgyBrDeWu8855aGA6LR9WHsyKn6bmXUQX1z+LXzuuvmxr8vFz3I77cOhPeGauPdtcKRBki1nwMJvGxzxcBESlTnBAPhjoXQQf3ECppRJ2OQ95u+DOlsfRL2nfl466oXX64OBNQpJi00kEoxa5dLdFmFUFQyQsi44wpaB/Wn43c5rX+1GthD+SyU8RKtxZgZEUMQpbjMSemOj/rHU99rVmN94BQ8Fqtt4F/AMEkEy+7Ks/uXesdG00F9n4XFWIBBJ2NOKNlYDoGQnBdeRxGwJLHjskAH9c2eRLfLLyfW4k9TWNTIZhzqo0AxGweopVZWKktYy1dmi0vhrWqPrDWJnAKydVDfrCZYR6EcM57qOO+2iNIeNKvda7eacD6v49xa/9HQU/FJM6bBhja/0P384xe6lzN1Ktma4us1SMm7lykTOKE4Q+UkZMs997Mf6Iynvjkmczos1yxrvZ2gyuzjJek8eCzOpzQOJJmMzLFWzHUSaCFn9oj1qT6hRTA7Ltx3UF9PANo70Ae6PpWMK/bVUEoFc4NTGXrVlufSDn2SVV+W38pzpWQfYpSq8H/sBFXkmOmVDGSEA9HRzGdPUbEfT31zMx+1DHlrY+SptETLrA1YBg+6PkhmIaWEWIBT+hTRma0Ftyct1KeBMeOpbzre0Rysdb2CuNI6X+Xw7kIOf3riS+Z+5XfHqkiUC2PHoIuQRQZmSGAI3kEoRcO2KXi8eFUv19UnNeE0jZ7jqW+8M9b6xzkcuRUPcsqIzNoWNXKK4awMdV4y74d0IYlIvLtbnvtdFQWeomSgEjtSXWyuT2tmQVHwsmoHsU7Zp10XLMZTP6mZfxh5qsa+xndrMXTi1lo2HgzyfypydOFU2AjG9So6DsuqkCXY0sJv6vQWahc+tYYTr1h+uz5jarZIdG63e6kDhjklY+s366HlX6Drwf3KDn26uPP6vxuLTTCVkT8NA3+WCwVKFhuUs7ooj0WBsgrBB5Ilyrgl6ZL1+VlhWlxhrzOpYDoYWxcMtfeW6pvOLNtxQzlMicBWKiPU3t5UQJKMxQSdVdEWt7htToEZplQh1Ff/JhVEB2PrglCtlFNffsbluyft0PXy4ZUTDl0hGilKLbwaTMRxNgsvCJ2syQ3BNlnIA2D8N8hCTyqLDsY2TWqpvT66NuV2yILJaHDByxxsfWOGPbICaXzC4kALtwXp1KlTtQuY/7Tm2DXo00igg51N0MfWQQcgDtzq1JxRjzOqCDYAo/CsyEeP2jH4NJhBpZD19kkcUKVfn8WUOKlSdDC26aGtigbrq83G4UnTSYdQ/1mj/MehTjFUJd5vY4BX5pVwvhcZkBYdCxaTEGqJ0Xtns60PFT8zf9rB2CZ/yliZJaVW/emG583f/MdlZU1WGEmJt8GLekNE1QGGWnnhkf/ccp7YZoVQO/edyHbGszO17eTAGFRnsEk7Z2wwTDFm5yXnu8WJEVqBdY7C5FrRwdimwT5ttGCNsnpak2YrZgeXSdT9AtY9hssGAYP03pst922fOF1WIIsTKE4HOxMrDkd6pmScj9IJlDlnxgEyOV+89WHbKw+PFcdMrjgdjG2GZ7bfnaD2ASihcK5YSsoVYq/GODFlhkbbJmIZtySMaWFJB2PrwsAmDC3XrOi0zNiincforQs5ihJIZVmojVJxflv9YYcq08HYZCrD3ow051HJFyaVtM8MpCMjJ8PJHGq5tU6FcnjLG1cvnZ7WkYxnZ7MjsW3u4SkcSXL1On3RjIrYhzGCjiZ5KrX3KOu8rQ2s3l2vmATqKFuaVCs6GNs08qUOXmwTzJ3sH78y7umYmBCtSpo421ASOPvKiE57HzVjp+CeWfDuYGzdoOZzgWVtrJtCtVMpyKFQOM0YojCg4ICpGWhJhYH5kSOlND3w7WBsY+NMLSTU/FlPGzNySNqCKuQ8uJRUTLqATrUBAQPDlz1DnA521j2TYc+Ec4gDvZ6J8zWRTLZC1Ho+e0WlswoRGUwUuQ3r1RlEUOeO19nrIKcFoeMZ21hsaGMp6qRlXDu+mw8AeH/hgz5vqeTvnxdCKvn4WtoH7XtfDnen/zYfJzC0h68chpALyRjO4kxw9fFXkUBySK0XcSOYZ+aeOhh7uuGZSdkqOgHzASKYnBES/xGcJ+8khOK9fGby6mBsOkiNZGVKnnUbMLtAzjkq0fg6dU9AfGZeu4OxyYQhAJRKktjlJf7MVI0pmeFJKtb5LbdfdymMDsY2CwPa2cWJlnR3Po6k3X99/Nrzrfml4YfvLb73H2uPuDotrSqRQb1PDOnrZx+kA0b2DOxgrMjU9M5nPGObCxuMjxhpH8NGj4uC77Xy339vZIdD0L8Mbrtd6pp3Q365OmRuQJZfrzxUDWBtsTaz4xcic/bMvrFYgdF6E9PIYuH0rqiDsc3IsuYJdqKE1iefUZliS9IZTSFfGJtEY2VIlnOnZ2aWHYxtTvup30c5waiEciHlGfNEDKzQIpMjjhoxlvTMhNHB2GT15YDWpIgCijVAxnpZvSF7R/BSU8jPrNjewdhmzXD9wiDjSXGcRJmhhHoWFIVh9GDAiCBifma4poOxzc7EztPUKc4ASkgFleSc0EbvXAg+Bp0jOFdyUdtOsXZoPx2MnYiWWyV+WrQMwhTOdoqwRqQgtESTtffoMrHTc+qZyauDsRP7A2p2YUY+2iFLJq8c43MEUspmTVKCQ4CiXIBtBTR1AHVEsmJBiGkLaB2MrRfQJPuZdkx/YCycNF39/7RGn3sDi/cWKPHWoh93/em5IgWCRcUZjWbwXkxGJ5TMyYLhjLo8uxRrPGO7d0VeBi34VyYyKFzRXgMY7+rEd599GXE6auYVINovO5srQOa0teniYiBG3ABIidW4vgkgbCETS6FS9LNTl/GMbU4idlPBSCLlyB6PQwWCVwoig/U6ytLXt4uyemby6mDspKS9do6jttMOXIEQnSCoTyoGz3GVg0YtsmCdN6tdGNH8V5uV63TbaWvAHYxtdEc1rNaJ/6tdVvfa9bu/nTSEtLU0D4cjHzfP/Yf2Y8MRyco7ZpwxR6PryMuYmCFRn6MohT96IL31iS6Yi60WZt20pfPxjK2LTbbrN/VdpGM9k0tiq0PijxajwG8PN5LqPNY/to/D6PBhaMsTTr9sM+eH68O3VxrI6mNJVmpdb2GStNbEzKlRMrxtOmf3zJKUDsY2tp7Wjlglpjo+goKcMrEJSG+DIpNIOgwlZP6PVBhxHDK8gICr8Wf37EwcDJVXaFPx9e1gCgWUqjOsFBDVmVXgR/ZMqYl6pjrYmVgQWUmXQWlkbKuS94mDMahsSnSSMRuN8Np2Jy3bHYxtTO7rnUjBigrm2LlQu179sDno+uWHT255NdlUD7SYGTUvKf5l47Vqz6m2wYjMErlcI4oF6XVi52cSBP3MPE4HY5MV9bP3IvqUiqKaFKXI0MTFaH1JUYmcO/p3JzlnHc/OxK1kytfXQyKx9GoJXBB/ko6BvqqXFCHu3bGMZmdiQTiy3qqkch2/a8jUpnLKnDRLCmS3DUyePO/qYOfknqBTCQIBiwcdheGVcNxkX197gYP2xukinh0y6WBsc9618BPQ4yewYI7RlSK1qRPQk3em3Wn2OkZ6Zm1zHYxN5jSj9M6l+uwaaCDUsWSoY4ozxSjGacZuqn4djO2zShqDquOsQdehmsj4ILmEEAQHtijQPMMWsfGMndyZesaC1z/8f+gH+EUtFwEA"
REAL = json.loads(gzip.decompress(base64.b64decode(_REAL_B64)))


def norm_name(s):
    s = re.sub(r"^(?:นางสาว|นาง|นาย|น\.ส\.|mr\.?|mrs\.?|ms\.?)\s*", "", (s or "").replace("...", "").strip(), flags=re.I)
    return re.sub(r"[\s.*]", "", s).lower()


# แบ่งตาม "ชื่อ": ชื่อที่ hash ลงกลุ่ม 0 เป็นชุดวัดผล -> ชื่อพวกนี้ไม่อยู่ในชุดเทรนเลย (ทั้งจริงและสังเคราะห์)
# ยกเว้นชื่อที่โผล่บ่อยมาก (เจ้าของบัญชี เช่น ผู้โอนซ้ำหลายสิบใบ) ให้อยู่ในชุดเทรน ไม่งั้นชุดวัดผลจะเอียงไปที่ชื่อเดียว
FREQ = Counter(norm_name(r["text"]) for r in REAL)


def is_val_name(text):
    k = norm_name(text)
    return FREQ[k] < 8 and int(hashlib.md5(k.encode()).hexdigest(), 16) % 4 == 0


files = sorted(p for p in IMAGE_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
by_md5 = {hashlib.md5(p.read_bytes()).hexdigest(): p for p in files}
real_train, real_val, missing = [], [], 0
for r in REAL:
    p = by_md5.get(r["md5"])
    if p is None:
        missing += 1
        continue
    x0, y0, x1, y1 = r["box"]
    img = Image.open(p).convert("RGB").crop((x0, y0, x1, y1))
    (real_val if is_val_name(r["text"]) else real_train).append((img, r["text"]))
VAL_NAMES = {norm_name(t) for _, t in real_val}
print("บรรทัดจริง: เทรน %d | วัดผล %d (ชื่อไม่ซ้ำ %d) | หารูปไม่เจอ %d" % (len(real_train), len(real_val), len(VAL_NAMES), missing))

In [ ]:
# รายชื่อสำหรับภาพสังเคราะห์: ชื่อ/นามสกุลไทยจาก pythainlp + ทับศัพท์อังกฤษ + ชื่อร้าน/บริษัท (ตัดชื่อที่อยู่ในชุดวัดผลออก)
from pythainlp.corpus import thai_words
try:
    from pythainlp.corpus import thai_female_names, thai_male_names, thai_family_names
    FIRST = sorted(set(thai_female_names()) | set(thai_male_names()))
    LAST = sorted(set(thai_family_names()))
except Exception as err:
    print("ไม่มีคลังชื่อใน pythainlp เวอร์ชันนี้ ใช้คำไทยทั่วไปแทน:", repr(err)[:80])
    FIRST, LAST = [], []
WORDS = [w for w in thai_words() if 2 <= len(w) <= 8 and re.fullmatch(r"[\u0e00-\u0e7f]+", w)]
if not FIRST:
    FIRST, LAST = WORDS[:20000], WORDS[20000:60000]
try:
    from pythainlp.transliterate import romanize
except Exception:
    romanize = None
print("ชื่อ %d | นามสกุล %d | คำทั่วไป %d | romanize: %s" % (len(FIRST), len(LAST), len(WORDS), romanize is not None))

PREFIX_TH = ["นาย ", "นาง ", "น.ส. ", "นางสาว ", "นาย", "น.ส.", ""]
PREFIX_EN = ["MR. ", "MRS. ", "MS. ", "Mr. ", "Ms. ", ""]


def rand_text():
    r = random.random()
    first, last = random.choice(FIRST), random.choice(LAST)
    if r < 0.55:                                                       # ชื่อไทย
        t = random.choice(PREFIX_TH) + first + " " + last
        k = random.random()
        if k < 0.15:
            t = random.choice(PREFIX_TH) + first + " " + last[0] + "."           # นามสกุลย่อ "แ."
        elif k < 0.25:
            t = random.choice(PREFIX_TH) + first + " " + last[0] + "***"         # ปิดนามสกุล "น***"
        return t
    if r < 0.80 and romanize is not None:                              # ชื่ออังกฤษ (ทับศัพท์)
        try:
            en = (romanize(first, engine="royin") + " " + romanize(last, engine="royin")).strip()
        except Exception:
            en = ""
        if en:
            en = en.upper() if random.random() < 0.6 else en.title()
            t = random.choice(PREFIX_EN) + en
            if random.random() < 0.3:                                       # ชื่อยาวถูกตัด "..."
                t = t[: random.randint(10, max(11, len(t) - 2))] + "..."
            return t
    k = random.random()                                                # ร้าน/บริษัท
    w = lambda: random.choice(WORDS)
    if k < 0.35:
        return "ร้าน" + w() + w()
    if k < 0.65:
        return "บริษัท " + w() + w() + " จำกัด" + (" (มหาชน)" if random.random() < 0.2 else "")
    return w() + w() + " " + w()


def render(text):
    font = ImageFont.truetype(random.choice(FONTS), random.randint(26, 46))
    l, t, r, b = font.getbbox(text)
    padx, pady = random.randint(4, 30), random.randint(4, 16)
    W, H = r - l + 2 * padx, b - t + 2 * pady
    bg = tuple(int(v) for v in np.clip(np.array([random.randint(225, 255)] * 3) + np.random.randint(-12, 12, 3), 0, 255))
    img = Image.new("RGB", (W, H), bg)
    d = ImageDraw.Draw(img)
    if random.random() < 0.3:                                          # ลายพื้นจางๆ แบบสลิป
        for _ in range(random.randint(3, 10)):
            c = tuple(int(v) for v in np.clip(np.array(bg) - random.randint(5, 25), 0, 255))
            x, y = random.randint(0, W), random.randint(0, H)
            d.ellipse((x - 30, y - 30, x + 30, y + 30), outline=c, width=2)
    fg = random.choice([(20, 20, 20), (40, 40, 40), (60, 60, 60), (30, 30, 80), (20, 60, 120), (90, 30, 30)])
    d.text((padx - l, pady - t), text, font=font, fill=fg)
    return augment(img)


def augment(img):
    if random.random() < 0.5:
        img = img.rotate(random.uniform(-1.5, 1.5), expand=True, fillcolor=img.getpixel((0, 0)))
    if random.random() < 0.5:
        s = random.uniform(0.45, 1.0)                                  # ย่อแล้วขยายกลับ = ความละเอียดต่ำแบบภาพจริง
        w, h = img.size
        img = img.resize((max(8, int(w * s)), max(8, int(h * s))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.0)))
    if random.random() < 0.5:
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(35, 90))
        img = Image.open(io.BytesIO(buf.getvalue())).convert("RGB")
    return img


synth = []
while len(synth) < N_SYNTH:
    t = rand_text()
    if norm_name(t) in VAL_NAMES or len(t) > 60:
        continue
    synth.append(t)
print("ตัวอย่างข้อความสังเคราะห์:", synth[:8])

In [ ]:
# ดูตัวอย่างภาพสังเคราะห์กับบรรทัดจริง (ตรวจว่าสระ/วรรณยุกต์วางถูกที่ ถ้าลอยผิดที่ = ไม่มี raqm)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(6, 2, figsize=(12, 9))
for i in range(6):
    ax[i, 0].imshow(render(synth[i])); ax[i, 0].axis("off")
    img, _t = random.choice(real_train)
    ax[i, 1].imshow(augment(img)); ax[i, 1].axis("off")
ax[0, 0].set_title("สังเคราะห์"); ax[0, 1].set_title("จริง")
plt.show()

In [ ]:
from transformers import (TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          default_data_collator)

processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)
tok = processor.tokenizer
cfg = model.config
if cfg.decoder_start_token_id is None:
    cfg.decoder_start_token_id = tok.cls_token_id if tok.cls_token_id is not None else tok.bos_token_id
if cfg.pad_token_id is None:
    cfg.pad_token_id = tok.pad_token_id
if cfg.eos_token_id is None:
    cfg.eos_token_id = tok.sep_token_id if tok.sep_token_id is not None else tok.eos_token_id
model.generation_config.max_length = MAX_LEN
print("start/pad/eos:", cfg.decoder_start_token_id, cfg.pad_token_id, cfg.eos_token_id)


class LineDataset(torch.utils.data.Dataset):
    def __init__(self, synth_texts, real_items, real_repeat):
        self.items = [("s", t) for t in synth_texts] + [("r", i) for i in range(len(real_items))] * real_repeat
        self.real = real_items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        kind, v = self.items[idx]
        if kind == "s":
            img, text = render(v), v
        else:
            img, text = self.real[v]
            img = augment(img)
        px = processor(images=img, return_tensors="pt").pixel_values[0]
        ids = tok(text, padding="max_length", max_length=MAX_LEN, truncation=True).input_ids
        return {"pixel_values": px, "labels": torch.tensor([i if i != tok.pad_token_id else -100 for i in ids])}


train_ds = LineDataset(synth, real_train, REAL_REPEAT)
print("ตัวอย่างเทรนต่อ epoch:", len(train_ds))

In [ ]:
# วัดผลกับบรรทัดจริงชุดวัดผล (ชื่อที่ไม่เคยเห็น): exact = ตรงหลังตัดคำนำหน้า/จุด/ช่องว่าง, kaggle = เกณฑ์ competition (เป็นส่วนหนึ่งกันได้), CER = อัตราตัวอักษรผิด
def cer(a, b):
    a, b = a or "", b or ""
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[len(b)] / max(len(b), 1)


@torch.no_grad()
def evaluate(m, items, show=0):
    m.eval()
    dev = next(m.parameters()).device
    preds = []
    for i in range(0, len(items), 16):
        px = processor(images=[img for img, _ in items[i:i + 16]], return_tensors="pt").pixel_values.to(dev)
        ids = m.generate(px, max_length=MAX_LEN)
        preds += [s.strip() for s in tok.batch_decode(ids, skip_special_tokens=True)]
    fix = lambda s: re.sub(r"([\u0e48-\u0e4b])\1+", r"\1", s.replace("\u0e4d\u0e32", "\u0e33"))
    exact = kag = 0
    c = []
    for (img, t), p in zip(items, preds):
        a, b = norm_name(fix(p)), norm_name(fix(t))
        exact += a == b
        kag += a != "" and (a == b or a in b or b in a)
        c.append(cer(fix(p), fix(t)))
    for (img, t), p in list(zip(items, preds))[:show]:
        print("  เฉลย %-35s | อ่านได้ %s" % (t, p))
    return {"n": len(items), "exact": exact, "kaggle": kag, "CER": round(float(np.mean(c)), 4)}


model.to("cuda" if torch.cuda.is_available() else "cpu")
before = evaluate(model, real_val, show=10)
print("ก่อน fine-tune:", before)

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/trocr_tmp", per_device_train_batch_size=BATCH, num_train_epochs=EPOCHS,
    learning_rate=LR, warmup_ratio=0.05, weight_decay=0.01, fp16=torch.cuda.is_available(),
    logging_steps=50, save_strategy="no", report_to="none", dataloader_num_workers=2, remove_unused_columns=False)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, data_collator=default_data_collator)
trainer.train()

In [ ]:
after = evaluate(model, real_val, show=20)
print("ก่อน:", before)
print("หลัง:", after)
model.save_pretrained(OUT_DIR)
processor.save_pretrained(OUT_DIR)
print("เซฟแล้ว ->", OUT_DIR, "| ใช้ใน notebook OCR: TROCR_MODEL = พาธโฟลเดอร์นี้ (หลัง Add Input เป็น Notebook Output)")